In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
nltk.download('punkt_tab')


import spacy


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/julienrm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df_fr = pd.read_csv('data/small_vocab_fr.txt', sep='\t', names=['text'])
df_fr.head()

,text
0,new jersey est parfois calme pendant l' automn...
1,les états-unis est généralement froid en juill...
2,"california est généralement calme en mars , et..."
3,"les états-unis est parfois légère en juin , et..."
4,"votre moins aimé fruit est le raisin , mais mo..."


In [3]:
df_en = pd.read_csv('data/small_vocab_en.txt', sep='\t', names=['text'])
df_en.head()

,text
0,"new jersey is sometimes quiet during autumn , ..."
1,the united states is usually chilly during jul...
2,"california is usually quiet during march , and..."
3,the united states is sometimes mild during jun...
4,"your least liked fruit is the grape , but my l..."


In [4]:
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text, remove_stopwords=False, language='french', remove_punctuation=False):
    """
    Preprocess text by cleaning and tokenizing
    """
    # Basic text cleaning
    text = text.strip() # .lower()
    def fix_punctuation_spacing(text):
        # Apostrophe: no spaces around it
        if text.find("'") != -1:
            text = text.replace(" '", "'").replace("' ", "'")

        # Hyphen in compound words: no spaces around it
        # Em dash (—) or en dash (–): space before and after for sentence breaks
        if text.find("-") != -1:
            # First handle spaced dashes (likely sentence breaks)
            text = text.replace(" - ", " — ")  # Convert to em dash
            text = text.replace(" -", " —").replace("- ", "— ")
            
            # Replace em dashes back to spaced format
            text = text.replace("—", " — ")
            
            # Clean up multiple spaces around em dashes
            text = re.sub(r'\s*—\s*', ' — ', text)
        
        # Comma: no space before, one space after
        if text.find(",") != -1:
            text = text.replace(" ,", ",")
            # Add space after comma if not already there
            text = re.sub(r',(?!\s)', ', ', text)
            # Fix multiple spaces after comma
            text = text.replace(",  ", ", ")

        # Period: no space before, one space after (except end of text)
        if text.find(".") != -1:
            text = text.replace(" .", ".")
            # Add space after period if not already there and not at end
            text = re.sub(r'\.(?!\s|$)', '. ', text)
            # Fix multiple spaces after period
            text = text.replace(".  ", ". ")
        
        # Semicolon: no space before, one space after
        if text.find(";") != -1:
            text = text.replace(" ;", ";")
            text = re.sub(r';(?!\s)', '; ', text)
            text = text.replace(";  ", "; ")
        
        # Colon: no space before, one space after
        if text.find(":") != -1:
            text = text.replace(" :", ":")
            text = re.sub(r':(?!\s)', ': ', text)
            text = text.replace(":  ", ": ")
        
        # Question mark: no space before, one space after
        if text.find("?") != -1:
            text = text.replace(" ?", "?")
            text = re.sub(r'\?(?!\s|$)', '? ', text)
            text = text.replace("?  ", "? ")
        
        # Exclamation mark: no space before, one space after
        if text.find("!") != -1:
            text = text.replace(" !", "!")
            text = re.sub(r'!(?!\s|$)', '! ', text)
            text = text.replace("!  ", "! ")
        
        # Opening parenthesis: one space before (if not at start), no space after
        if text.find("(") != -1:
            text = re.sub(r'(?<!\s)(?<!^)\(', ' (', text)  # Add space before if not already there
            text = text.replace("( ", "(")  # Remove space after
            text = text.replace("  (", " (")  # Fix double spaces
        
        # Closing parenthesis: no space before, one space after (if not at end)
        if text.find(")") != -1:
            text = text.replace(" )", ")")
            text = re.sub(r'\)(?!\s|$|[.,;:!?])', ') ', text)  # Add space after unless at end or before punctuation
            text = text.replace(")  ", ") ")
        
        # Clean up any multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

    text = fix_punctuation_spacing(text)

    # Remove digits
    # cleaned_text = ''.join(char for char in text if not char.isdigit())
    
    # Remove punctuation
    # if remove_punctuation:
    #     for punctuation in string.punctuation:
    #         cleaned_text = cleaned_text.replace(punctuation, ' ')

    # Tokenize
    word_tokens = word_tokenize(text, language=language)
    
    # if asked to remove stopwords
    if remove_stopwords:
        print("Removing stopwords")
        # Remove stop words
        if language == 'french':
            stop_words = set(stopwords.words('french'))
        else:
            stop_words = set(stopwords.words('english'))
    
        tokens_cleaned = [w for w in word_tokens if w not in stop_words and len(w) > 0]
    
        return tokens_cleaned
    
    # # Load relevant language model
    # if language == 'french':
        
    #     nlp = spacy.load('fr_core_news_sm')
    # else:
    #     nlp = spacy.load('en_core_web_sm')

    # def process_text(text):
    #     # this is processing part.
    #     doc = nlp(text)

    #     # Filtering step
    #     filtered_tokens = [token.text for token in doc if not token.is_stop]

    #     print("Filtered Tokens:", filtered_tokens)
    word_tokens = [wt for wt in word_tokens if len(wt) > 0]
    # print(word_tokens)

    return word_tokens

# Apply preprocessing to French data
df_fr['tokens'] = df_fr['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='french'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_fr = TfidfVectorizer()
# Training it on the texts
weighted_df_fr = pd.DataFrame(tf_idf_vectorizer_fr.fit_transform(df_fr['text']).toarray(),
                    columns = tf_idf_vectorizer_fr.get_feature_names_out())

# Apply preprocessing to English data
df_en['tokens'] = df_en['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='english'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_en = TfidfVectorizer()

# Training it on the texts
weighted_df_en = pd.DataFrame(tf_idf_vectorizer_en.fit_transform(df_en['text']).toarray(),
                    columns = tf_idf_vectorizer_en.get_feature_names_out())

print("French preprocessing complete")
print("English preprocessing complete")

# Rename columns to be specific to each language
df_fr_renamed = df_fr.rename(columns={'text': 'text_fr', 'tokens': 'tokens_fr'})
df_en_renamed = df_en.rename(columns={'text': 'text_en', 'tokens': 'tokens_en'})

# Combine side by side
df_fr_en = pd.concat([df_fr_renamed, df_en_renamed], axis=1)

import csv
with open('data/cleaned_texts.csv', 'w', newline='') as csvfile:
    df_fr_en.to_csv(csvfile, index=False)

French preprocessing complete
English preprocessing complete


In [5]:
print("\nFrench:", weighted_df_fr.shape[1], "\nEnglish:", weighted_df_en.shape[1])


French: 321 
English: 196


In [6]:
# Combine side by side

df_fr_en

,text_fr,tokens_fr,text_en,tokens_en
0,new jersey est parfois calme pendant l' automn...,"[new, jersey, est, parfois, calme, pendant, l'...","new jersey is sometimes quiet during autumn , ...","[new, jersey, is, sometimes, quiet, during, au..."
1,les états-unis est généralement froid en juill...,"[les, états-unis, est, généralement, froid, en...",the united states is usually chilly during jul...,"[the, united, states, is, usually, chilly, dur..."
2,"california est généralement calme en mars , et...","[california, est, généralement, calme, en, mar...","california is usually quiet during march , and...","[california, is, usually, quiet, during, march..."
3,"les états-unis est parfois légère en juin , et...","[les, états-unis, est, parfois, légère, en, ju...",the united states is sometimes mild during jun...,"[the, united, states, is, sometimes, mild, dur..."
4,"votre moins aimé fruit est le raisin , mais mo...","[votre, moins, aimé, fruit, est, le, raisin, ,...","your least liked fruit is the grape , but my l...","[your, least, liked, fruit, is, the, grape, ,,..."
...,...,...,...,...
137855,"la france est jamais occupée en mars , et il e...","[la, france, est, jamais, occupée, en, mars, ,...","france is never busy during march , and it is ...","[france, is, never, busy, during, march, ,, an..."
137856,"l' inde est parfois belle au printemps , et il...","[l'inde, est, parfois, belle, au, printemps, ,...","india is sometimes beautiful during spring , a...","[india, is, sometimes, beautiful, during, spri..."
137857,"l' inde est jamais mouillé pendant l' été , ma...","[l'inde, est, jamais, mouillé, pendant, l'été,...","india is never wet during summer , but it is s...","[india, is, never, wet, during, summer, ,, but..."
137858,"la france est jamais froid en janvier , mais i...","[la, france, est, jamais, froid, en, janvier, ...","france is never chilly during january , but it...","[france, is, never, chilly, during, january, ,..."


In [7]:
df_fr_en.text_fr[137855]

'la france est jamais occupée en mars , et il est parfois agréable en septembre .'

In [8]:
df_fr_en.tokens_fr[137855]

['la',
 'france',
 'est',
 'jamais',
 'occupée',
 'en',
 'mars',
 ',',
 'et',
 'il',
 'est',
 'parfois',
 'agréable',
 'en',
 'septembre',
 '.']

In [9]:
weighted_df_en

,am,and,animal,animals,apple,apples,april,are,aren,august,...,when,where,white,why,winter,wonderful,would,yellow,you,your
0,0.0,0.180260,0.0,0.0,0.000000,0.0,0.366932,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.162972,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.175169,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.175852,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.000000,0.0,0.0,0.304045,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.255303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137855,0.0,0.185612,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137856,0.0,0.191898,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137857,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.377138,0.0,0.0,0.0,0.0,0.000000
137858,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000


In [10]:
df_fr_en['text_fr'][23]

"paris est doux pendant l' été , mais il est généralement occupé en avril ."

In [11]:
df_fr_en['text_fr']
df_fr_en['text_en']

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

In [12]:
df_fr_en["tokens_fr"]

0         [new, jersey, est, parfois, calme, pendant, l'...
1         [les, états-unis, est, généralement, froid, en...
2         [california, est, généralement, calme, en, mar...
3         [les, états-unis, est, parfois, légère, en, ju...
4         [votre, moins, aimé, fruit, est, le, raisin, ,...
                                ...                        
137855    [la, france, est, jamais, occupée, en, mars, ,...
137856    [l'inde, est, parfois, belle, au, printemps, ,...
137857    [l'inde, est, jamais, mouillé, pendant, l'été,...
137858    [la, france, est, jamais, froid, en, janvier, ...
137859    [l'orange, est, son, fruit, préféré, ,, mais, ...
Name: tokens_fr, Length: 137860, dtype: object

In [13]:
from LSTM_translator import train_translator_from_tokens, load_translator_for_inference, test_translation

In [14]:

import tensorflow as tf

In [15]:
# Check current setup
print("TensorFlow version:", tf.__version__)
print("Built with MPS:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.16.2
Built with MPS: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [16]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [ ]:
# # Your data structure: df with columns ['tokens_fr', 'tokens_en']
# # Example: df.iloc[0]['tokens_fr'] = ['new', 'jersey', 'est', 'parfois', 'calme', ...]

# tf.random.set_seed(42)

# # Train the model
# # with tf.device('/GPU:0'):
# with tf.device('/CPU:0'):
#     translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# # Save for later use
# translator.save_model('lstm_translator')

# # Test some translations
# test_translation(translator, df_fr_en, n_examples=5)


=== Training Configuration ===
Embedding Dim: 512
Hidden Units: 1024
Max Vocab Size: 50000
Attention: True
Bidirectional: True
Teacher Forcing: True
Scheduled Sampling: True
Epochs: 18, Patience: 10
Processing tokenized data...
Dataset size after filtering: 137860


2025-09-09 21:38:14.559836: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-09-09 21:38:14.559862: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-09-09 21:38:14.559865: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-09-09 21:38:14.559880: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-09 21:38:14.559888: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Training set: 109488
Test set: 27372
Validation set: 1000
Building vocabularies...
French vocabulary size: 359
English vocabulary size: 204
Converting tokens to sequences...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Starting training for maximum 18 epochs with patience 10...
Using batch_size=64, attention=True, bidirectional=True
Teacher forcing=True, scheduled_sampling=True
Using scheduled sampling training...
Note: Scheduled sampling is enabled but using simplified version.
For full scheduled sampling, consider using a custom training loop.
Epoch 1/18
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 2223s 1s/step - accuracy: 0.9137 - loss: 0.2902 - val_accuracy: 0.9191 - val_loss: 0.2462 - learning_rate: 0.0010
Epoch 2/18
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 64940s 38s/step - accuracy: 0.9348 - loss: 0.1981 - val_accuracy: 0.9458 - val_loss: 0.1683 - learning_rate: 0.0010
Epoch 3/18
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 2256s 1s/step - accuracy: 0.9569 - l


Test Loss: 0.0097
Test Accuracy: 0.9942
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Saving model to lstm_translator...
Model saved successfully!

TESTING TRANSLATIONS


2025-09-11 04:56:32.553874: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.



French tokens: ['new', 'jersey', 'est', 'jamais', 'belle', 'au', 'mois', "d'août", ',', 'mais', 'il', 'est', 'généralement', 'froid', 'en', 'juin', '.']
True English: ['new', 'jersey', 'is', 'never', 'beautiful', 'during', 'august', ',', 'but', 'it', 'is', 'usually', 'chilly', 'in', 'june', '.']
Predicted: ['new', 'jersey', 'is', 'never', 'beautiful', 'during', 'august', ',', 'but', 'it', 'is', 'usually', 'cold', 'in', 'june', '.']
--------------------------------------------------

French tokens: ['paris', 'est', 'relaxant', 'parfois', 'pendant', "l'hiver", ',', 'mais', 'il', 'est', 'généralement', 'beau', 'en', 'avril', '.']
True English: ['paris', 'is', 'sometimes', 'relaxing', 'during', 'winter', ',', 'but', 'it', 'is', 'usually', 'beautiful', 'in', 'april', '.']
Predicted: ['paris', 'is', 'sometimes', 'relaxing', 'during', 'winter', ',', 'but', 'it', 'is', 'usually', 'beautiful', 'in', 'april', '.']
--------------------------------------------------

French tokens: ['votre', 'fru

In [29]:
translator = load_translator_for_inference('lstm_translator')

# # Method 1: Preprocess then translate tokens manually
# french_tokens = translator.preprocess_french_phrase("Bonjour, comment allez-vous ?")
# english_tokens = translator.translate_tokens(french_tokens)
# print(english_tokens)

# Method 2: Translate entire sentence directly
english_tokens = translator.translate_sentence("il aime la mangue ?")
print(english_tokens)

Loading model from lstm_translator...
Error loading main model: Missing required positional argument
Rebuilding model from scratch...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Inference models not found or corrupted, will build when needed.
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['he', 'likes', 'a', 'rusty', 'black', 'truck', '.']


In [19]:
# After kernel restart - Fresh test
from LSTM_translator import load_translator_for_inference

translator_loaded = load_translator_for_inference('lstm_translator')
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
result = translator_loaded.translate_tokens(test_tokens)
print(result)

Loading model from lstm_translator...
Error loading main model: Missing required positional argument
Rebuilding model from scratch...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Inference models not found or corrupted, will build when needed.
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['new', 'jersey', 'is', 'quiet', 'during', 'spring', '.']


In [20]:
# Test cell - Loading saved model and attempting prediction
import tensorflow as tf
from LSTM_translator import load_translator_for_inference
import numpy as np
# Load the saved model 
translator_loaded = load_translator_for_inference('lstm_translator')
# Test with a simple French sentence
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
print(f"Testing translation of: {test_tokens}")
# Try to translate (this should fail with same error)
try:
    result = translator_loaded.translate_tokens(test_tokens)
    print(f"Translation: {result}")
except AttributeError as e:
    print(f"Expected error: {e}")
    print("Confirming the issue exists with loaded models too")


Loading model from lstm_translator...
Error loading main model: Missing required positional argument
Rebuilding model from scratch...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Inference models not found or corrupted, will build when needed.
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
Testing translation of: ['new', 'jersey', 'est', 'parfois', 'calme']


Translation: ['new', 'jersey', 'is', 'quiet', 'during', 'spring', '.']


In [21]:

translator = load_translator_for_inference('lstm_translator')

# Translate new French tokens but change to words for vocabulary
french_tokens = ['bonjour', 'comment', 'allez', 'vous']
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['hello', 'how', 'are', 'you']
french_tokens = ["C'est meilleur la banane ?"]
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['Is the banana better?']

Loading model from lstm_translator...
Error loading main model: Missing required positional argument
Rebuilding model from scratch...
Building the model with attention=True, bidirectional=True...
Model built with 29,875,916 parameters
Inference models not found or corrupted, will build when needed.
Building inference models...
⚠️  Complex architecture detected (bidirectional/attention).
Skipping inference model building - use main model for translation.
Model loaded successfully!
['rabbits', 'were', 'your', 'favorite', 'animals', '.']
['mice', 'are', 'her', 'favorite', 'animals', '.']


In [34]:
# Method 2: Translate entire sentence directly

print(translator.translate_sentence("c'est bien la banane ?"))
print(translator.translate_sentence("parfois calme parfois neigeux"))

print(translator.translate_sentence("new jersey est parfois calme pendant l' automne , et il est neigeux en avril ."))
print(translator.translate_sentence("le pamplemousse est votre fruit le plus aimé , mais le raisin est leur plus aimé ."))

['he', 'wanted', 'to', 'go', 'to', 'the', 'united', 'states', 'last', 'summer', '.']
['he', 'is', 'sometimes', 'quiet', 'during', 'spring', '.']
['when', 'might', 'go', 'to', 'india', 'last', 'august', '?']
['the', 'grapefruit', 'is', 'your', 'most', 'loved', 'fruit', ',', 'but', 'the', 'grape', 'is', 'their', 'most', 'loved', '.']


In [23]:
df_fr_en['text_fr'][14230]

'le pamplemousse est votre fruit le plus aimé , mais le raisin est leur plus aimé .'

In [31]:
df_fr_en["text_en"]

0         new jersey is sometimes quiet during autumn , ...
1         the united states is usually chilly during jul...
2         california is usually quiet during march , and...
3         the united states is sometimes mild during jun...
4         your least liked fruit is the grape , but my l...
                                ...                        
137855    france is never busy during march , and it is ...
137856    india is sometimes beautiful during spring , a...
137857    india is never wet during summer , but it is s...
137858    france is never chilly during january , but it...
137859    the orange is her favorite fruit , but the ban...
Name: text_en, Length: 137860, dtype: object

In [32]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_en"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

is: 205858
,: 140897
.: 129039
in: 75525
it: 75137
during: 74933
the: 67628
but: 63987
and: 59850
sometimes: 37746
usually: 37507
never: 37500
least: 27564
favorite: 27371
fruit: 27105
most: 14934
loved: 13666
liked: 13546
new: 12197
paris: 11334
india: 11277
united: 11270
states: 11270
california: 11250
jersey: 11225
france: 11170
china: 10953
he: 10786
she: 10786
grapefruit: 10118
your: 9734
my: 9700
his: 9700
her: 9700
fall: 9134
june: 9133
spring: 9102
january: 9090
winter: 9038
march: 9023
autumn: 9004
may: 8995
nice: 8984
september: 8958
july: 8956
april: 8954
november: 8951
summer: 8948
december: 8945
february: 8942
our: 8932
their: 8932
freezing: 8928
pleasant: 8916
beautiful: 8915
october: 8910
snowy: 8898
warm: 8890
cold: 8878
wonderful: 8808
dry: 8794
busy: 8791
august: 8789
chilly: 8770
rainy: 8761
mild: 8743
wet: 8726
relaxing: 8696
quiet: 8693
hot: 8639
dislikes: 7314
likes: 7314
limes: 5554
mangoes: 5549
lemons: 5533
grapes: 5525
apples: 5452
oranges: 5452
strawberries: 

In [33]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_fr"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

est: 196809
.: 135619
,: 123135
en: 105768
il: 84079
les: 65255
mais: 63987
et: 59851
la: 49861
parfois: 37746
jamais: 37215
le: 35306
l': 32917
généralement: 31292
moins: 27557
au: 25738
aimé: 24842
fruit: 23626
préféré: 22886
agréable: 17751
froid: 16794
son: 16496
chaud: 16405
de: 15070
plus: 14934
automne: 14727
mois: 14350
à: 13870
elle: 12056
citrons: 11679
paris: 11334
inde: 11277
états-unis: 11210
france: 11170
jersey: 11052
new: 11047
chine: 10936
pendant: 10741
pamplemousse: 10140
mon: 9403
votre: 9368
juin: 9133
printemps: 9100
janvier: 9090
hiver: 9038
mars: 9023
été: 8999
mai: 8995
septembre: 8958
juillet: 8956
avril: 8954
novembre: 8951
décembre: 8945
février: 8942
octobre: 8911
aime: 8870
août: 8789
merveilleux: 8704
relaxant: 8458
doux: 8458
humide: 8446
notre: 8319
californie: 8189
sec: 7957
leur: 7855
occupé: 7782
pluvieux: 7658
calme: 7256
beau: 6387
habituellement: 6215
pommes: 5844
pêches: 5844
oranges: 5844
poires: 5844
fraises: 5844
bananes: 5844
verts: 5835
rais

In [ ]:
df_fr_en["text_en"]

In [ ]:
from RNN_translator_mps import RNNTranslator, grid_search_rnn

# Define parameter grid
param_grid = {
    'embedding_dim': [64, 128],
    'hidden_units': [128, 256],
    'dropout_rate': [0.1, 0.2],
    'learning_rate': [0.001, 0.0001]
}
# Run grid search
best_configs = grid_search_rnn(df_fr_en, param_grid, epochs=10, n_best=3)
# Train with best config
best_params = best_configs[0]['params']
rnn = RNNTranslator(**best_params)

Starting RNN Grid Search...
Parameter grid: {'embedding_dim': [64, 128], 'hidden_units': [128, 256], 'dropout_rate': [0.1, 0.2], 'learning_rate': [0.001, 0.0001]}
Total configurations to test: 16

Configuration 1/16
Parameters: {'dropout_rate': 0.1, 'embedding_dim': 64, 'hidden_units': 128, 'learning_rate': 0.001}
Processing tokenized data...
Dataset size after filtering: 137860
Training set: 109288
Test set: 27572
Validation set: 1000
Building vocabularies...
French vocabulary size: 361
English vocabulary size: 204
Converting tokens to sequences...
Building simple RNN model...
Model built with 111,884 parameters
Starting training for maximum 10 epochs with patience 3...
Epoch 1/10
  92/1708 ━━━━━━━━━━━━━━━━━━━━ 31:02 1s/step - accuracy: 0.2862 - loss: 3.8979

KeyboardInterrupt: 